In [1]:
import pandas as pd, numpy as np
df = pd.read_csv("data/Dataset.csv")
df = df.replace("Not Given", "Unknown")
df = df.drop_duplicates(subset=["title", "type", "release_year"]).reset_index(drop=True)
df["date_added"]   = pd.to_datetime(df["date_added"], format="mixed")
df["year_added"]   = df["date_added"].dt.year
df["month_added"]  = df["date_added"].dt.month
df["duration_num"] = df["duration"].str.extract(r"(\d+)").astype(int)
df["main_country"] = df["country"].str.split(",").str[0].str.strip()
df["genres"]       = df["listed_in"].str.split(", ")

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df["soup"] = df["listed_in"] + " " + df["director"] + " " + df["country"] + " " + df["rating"] + " " + df["type"]
X = TfidfVectorizer(stop_words="english").fit_transform(df["soup"])
idx = pd.Series(df.index, index=df["title"])

def recommend(title, n=10):
    i = idx[title]
    scores = cosine_similarity(X[i], X).ravel()
    top = scores.argsort()[::-1][1:n+1]
    return df.iloc[top][["title", "type", "listed_in", "rating"]].assign(score=scores[top].round(3))

recommend("Ganglands")

,title,type,listed_in,rating,score
1182,Sentinelle,Movie,"Action & Adventure, Dramas, International Movies",TV-MA,0.847
2141,Earth and Blood,Movie,"Dramas, International Movies, Thrillers",TV-MA,0.773
8405,Crime Time,TV Show,"Crime TV Shows, International TV Shows, TV Act...",TV-MA,0.685
6735,Lupin,TV Show,"Crime TV Shows, International TV Shows, TV Act...",TV-MA,0.685
7497,Ad Vitam,TV Show,"Crime TV Shows, International TV Shows, TV Dramas",TV-MA,0.617
7662,Black Spot,TV Show,"Crime TV Shows, International TV Shows, TV Dramas",TV-MA,0.617
8061,The Frozen Dead,TV Show,"Crime TV Shows, International TV Shows, TV Dramas",TV-MA,0.617
8010,The Chalet,TV Show,"Crime TV Shows, International TV Shows, TV Dramas",TV-MA,0.617
8383,Cannabis,TV Show,"Crime TV Shows, International TV Shows, TV Dramas",TV-MA,0.617
8062,La Mante,TV Show,"Crime TV Shows, International TV Shows, TV Dramas",TV-MA,0.617


In [1]:
import difflib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ---------- Output folders ----------
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


# ---------- 1. Load and clean dataset ----------
df = pd.read_csv("data/Dataset.csv")

df = df.fillna("Unknown")
df = df.replace("Not Given", "Unknown")

df = df.drop_duplicates(
    subset=["title", "type", "release_year"]
).reset_index(drop=True)

df["title"] = df["title"].astype(str)
df["type"] = df["type"].astype(str)


def split_list(text):
    """Split comma-separated values and remove empty/unknown entries."""
    return [
        part.strip()
        for part in str(text).split(",")
        if part.strip()
        and part.strip().lower() not in ("unknown", "nan")
    ]


df["genres"] = df["listed_in"].apply(split_list)
df["countries"] = df["country"].apply(split_list)
df["directors"] = df["director"].apply(split_list)


# ---------- 2. Build content tags ----------
def tag(values, prefix):
    """Convert values into individual searchable tags."""
    return [
        prefix + value.replace(" ", "")
                     .replace("&", "and")
                     .lower()
        for value in values
    ]


def build_soup(row):
    # Give genres additional importance
    tags = tag(row["genres"], "g_") * 2
    tags += tag(row["directors"], "d_")
    tags += tag(row["countries"], "c_")

    rating = str(row["rating"]).strip().lower()
    content_type = str(row["type"]).strip().replace(" ", "").lower()

    tags.append("r_" + rating)
    tags.append("t_" + content_type)

    return " ".join(tags)


df["soup"] = df.apply(build_soup, axis=1)


# ---------- 3. Convert tags into TF-IDF features ----------
tfidf = TfidfVectorizer(
    token_pattern=r"\S+",
    lowercase=False
)

X = tfidf.fit_transform(df["soup"])

# Map titles to row indices.
# If duplicate titles exist, use the first matching record.
title_to_idx = {}

for i, title in enumerate(df["title"]):
    title_to_idx.setdefault(title.strip().casefold(), i)


# ---------- Figure helper: recommendations ----------
def save_recommendation_figure(source_title, recommendations):
    """Save a horizontal bar chart of recommendation similarity scores."""
    if recommendations.empty:
        return

    plot_df = recommendations.sort_values("similarity", ascending=True)

    fig_height = max(4.5, 0.48 * len(plot_df) + 1.5)
    fig, ax = plt.subplots(figsize=(11, fig_height))

    bars = ax.barh(plot_df["title"], plot_df["similarity"])
    ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=9)

    ax.set_title(f"Top Recommendations for: {source_title}", pad=14)
    ax.set_xlabel("Cosine Similarity Score")
    ax.set_ylabel("Recommended Title")
    ax.set_xlim(0, max(1.0, float(plot_df["similarity"].max()) * 1.18))
    ax.grid(axis="x", linestyle="--", alpha=0.35)
    ax.set_axisbelow(True)

    fig.tight_layout()
    safe_name = "".join(
        char if char.isalnum() else "_"
        for char in source_title.lower()
    ).strip("_")
    fig.savefig(FIGURES_DIR / f"recommendations_{safe_name}.png", dpi=200,
                bbox_inches="tight")
    plt.close(fig)


# ---------- 4. Recommendation function ----------
def recommend(title, n=10):
    """Recommend similar Netflix titles using cosine similarity."""
    key = str(title).strip().casefold()

    if key not in title_to_idx:
        close = difflib.get_close_matches(
            key,
            list(title_to_idx.keys()),
            n=3,
            cutoff=0.5
        )

        suggestions = [
            df.iloc[title_to_idx[name]]["title"]
            for name in close
        ]

        raise KeyError(
            f"Title '{title}' not found. "
            f"Did you mean: {suggestions}?"
        )

    idx = title_to_idx[key]

    scores = cosine_similarity(X[idx], X).ravel()
    scores[idx] = -1

    n = max(0, min(int(n), len(df) - 1))
    top_indices = np.argsort(scores)[::-1][:n]

    output = df.iloc[top_indices][
        ["title", "type", "listed_in", "rating"]
    ].copy()

    output["similarity"] = scores[top_indices].round(3)
    output = output.reset_index(drop=True)

    # Save a figure for this source title
    save_recommendation_figure(title, output)

    return output


# Display recommendations (same console output as before)
sample_titles = [
    "Ganglands",
    "Midnight Mass",
    "The Great British Baking Show",
    "Motu Patlu in Wonderland"
]

for title in sample_titles:
    print(f"\n=== Because you watched: {title} ===")

    try:
        print(recommend(title, 8).to_string(index=False))
    except KeyError as error:
        print(error)


# ---------- 5. Evaluation against random baseline ----------
def jaccard(a, b):
    """Measure genre overlap between two titles."""
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a | b) else 0.0


rng = np.random.default_rng(42)

sample_size = min(300, len(df))
sample = rng.choice(len(df), sample_size, replace=False)

similarity_matrix = cosine_similarity(X[sample], X)

rows = []

for source_idx, scores in zip(sample, similarity_matrix):
    scores = scores.copy()
    scores[source_idx] = -1

    top_n = min(10, len(df) - 1)
    model_indices = np.argsort(scores)[::-1][:top_n]

    candidates = np.delete(np.arange(len(df)), source_idx)
    random_indices = rng.choice(
        candidates,
        size=top_n,
        replace=False
    )

    source_genres = df.at[source_idx, "genres"]

    for method, indices in [
        ("Model", model_indices),
        ("Random", random_indices)
    ]:
        genre_jaccard_scores = []
        genre_hits = []

        for recommended_idx in indices:
            recommended_genres = df.at[recommended_idx, "genres"]

            genre_jaccard_scores.append(
                jaccard(source_genres, recommended_genres)
            )

            genre_hits.append(
                bool(set(source_genres) & set(recommended_genres))
            )

        rows.append({
            "method": method,
            "genre_jaccard": np.mean(genre_jaccard_scores),
            "genre_hit_rate": np.mean(genre_hits)
        })


result = (
    pd.DataFrame(rows)
    .groupby("method")[["genre_jaccard", "genre_hit_rate"]]
    .mean()
    .round(3)
)

print("\n=== Evaluation Results ===")
print("Evaluation: genre overlap against a random baseline")
print(result.to_string())


# ---------- 6. Figures for evaluation ----------
# Figure 1: Genre Jaccard comparison
fig, ax = plt.subplots(figsize=(7, 5))
values = result["genre_jaccard"]
bars = ax.bar(values.index, values.values)
ax.bar_label(bars, fmt="%.3f", padding=4)
ax.set_title("Genre Jaccard: Model vs Random Baseline")
ax.set_ylabel("Mean Genre Jaccard Score")
ax.set_ylim(0, max(0.1, float(values.max()) * 1.2))
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "evaluation_genre_jaccard.png", dpi=200,
            bbox_inches="tight")
plt.close(fig)

# Figure 2: Genre hit-rate comparison
fig, ax = plt.subplots(figsize=(7, 5))
values = result["genre_hit_rate"]
bars = ax.bar(values.index, values.values)
ax.bar_label(bars, fmt="%.3f", padding=4)
ax.set_title("Genre Hit Rate: Model vs Random Baseline")
ax.set_ylabel("Mean Genre Hit Rate")
ax.set_ylim(0, min(1.0, max(0.1, float(values.max()) * 1.2)))
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "evaluation_genre_hit_rate.png", dpi=200,
            bbox_inches="tight")
plt.close(fig)

# Figure 3: Both evaluation metrics in one grouped chart
metric_plot = result.rename(columns={
    "genre_jaccard": "Genre Jaccard",
    "genre_hit_rate": "Genre Hit Rate"
})
ax = metric_plot.plot(kind="bar", figsize=(8, 5))
ax.set_title("Content-Based Recommendations vs Random Baseline")
ax.set_xlabel("Recommendation Method")
ax.set_ylabel("Mean Score")
ax.set_ylim(0, 1)
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.set_axisbelow(True)
ax.legend(title="Evaluation Metric")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "evaluation_metrics_comparison.png", dpi=200,
            bbox_inches="tight")
plt.close()


# ---------- 7. Print model information (same console output) ----------
print("\n=== Model Information ===")
print("Recommendation Type: Content-Based Filtering")
print("Feature Extraction: TF-IDF Vectorization")
print("Similarity Measure: Cosine Similarity")

print("\n=== Evaluation Interpretation ===")
print("Genre Jaccard: Measures genre-set overlap.")
print("Genre Hit Rate: Measures recommendations sharing at least one genre.")
print("Baseline: Randomly selected titles, excluding the source title.")
print("Evaluation Type: Genre-overlap performance.")
print("Note: These metrics do not measure overall recommendation accuracy.")

print(f"\nFigures saved in: {FIGURES_DIR.resolve()}")
print("Recommendation charts: recommendations_<title>.png")
print("Evaluation charts: evaluation_genre_jaccard.png, "
      "evaluation_genre_hit_rate.png, evaluation_metrics_comparison.png")



=== Because you watched: Ganglands ===
                 title    type                                                     listed_in rating  similarity
                 Lupin TV Show Crime TV Shows, International TV Shows, TV Action & Adventure  TV-MA       0.865
            Crime Time TV Show Crime TV Shows, International TV Shows, TV Action & Adventure  TV-MA       0.865
         Fatal Destiny TV Show Crime TV Shows, International TV Shows, TV Action & Adventure  TV-MA       0.789
          Agent Raghav TV Show Crime TV Shows, International TV Shows, TV Action & Adventure  TV-14       0.782
               Smoking TV Show Crime TV Shows, International TV Shows, TV Action & Adventure  TV-MA       0.782
The Eagle of El-Se'eed TV Show Crime TV Shows, International TV Shows, TV Action & Adventure  TV-14       0.771
             The Truth TV Show Crime TV Shows, International TV Shows, TV Action & Adventure  TV-14       0.771
          Monkey Twins TV Show Crime TV Shows, International TV 